# SINAN Dengue — download e análise da base inteira

**Se o CSV já está no Colab, mantenha `EXECUTAR_DOWNLOAD = False`.** A execução começa diretamente da preparação analítica. Nenhuma célula apaga DataFrames existentes ou os arquivos de origem.

O download, quando habilitado, continua baixando tudo do zero. A análise é independente: o CSV é convertido uma vez para Parquet e todas as consultas usam uma conexão comum. Não se carrega a base inteira em pandas. As tabelas exibidas são resultados agregados da base completa, sem percentual de amostragem.

A primeira conversão ainda precisa ler os 10+ GiB do CSV. Depois, as consultas leem apenas colunas necessárias; município e região usam um resumo exato já calculado. Não há promessa de tempo fixo. CPU é suficiente; GPU não acelera estas consultas.


In [ ]:
from pathlib import Path
import os
import shutil

# FLUXO: False para analisar o CSV que já está no ambiente.
EXECUTAR_DOWNLOAD = False
PREPARAR_ANALISE = True  # False abre explicitamente PARQUET_EXISTENTE.
THREADS_DOWNLOAD = 20
THREADS_ANALISE = min(8, os.cpu_count() or 2)
MEMORIA_ANALISE = "3GB"
SALVAR_NO_DRIVE = False
PASTA_NO_DRIVE = "Denguinhos/datasets"

TRABALHO = Path("/content/sinan-dengue")
TRABALHO.mkdir(parents=True, exist_ok=True)
SCRIPT = TRABALHO / "baixar-sinan-dengue-arquivos.py"
DADOS = TRABALHO / "dados"
# Preserva o caminho da sessão existente. Edite se o arquivo estiver em outra pasta.
CSV_FINAL = Path(globals().get("CSV_FINAL", DADOS / "dengue-completo.csv"))
PASTA_ANALISE = TRABALHO / "analise"
PARQUET_EXISTENTE = PASTA_ANALISE / "dengue-analitico.parquet"

CODIGO_MUNICIPIO = "2611606"  # Recife; aceita 6 ou 7 dígitos.
DATA_SERIE = "DT_NOTIFIC"  # ou DT_SIN_PRI
PERIODO_SERIE = "month"  # day, week, month ou year
CALCULAR_CORRELACOES = True
CAMPOS_CORRELACAO = ["FEBRE", "MIALGIA", "CEFALEIA", "DIABETES", "HIPERTENSA"]
CALCULAR_DUPLICATAS_EXATAS = False  # operação pesada e optativa

print("CSV:", CSV_FINAL)
print(f"Disco livre: {shutil.disk_usage(TRABALHO).free / 1024**3:.2f} GiB")
print("Download:", EXECUTAR_DOWNLOAD, "| preparação analítica:", PREPARAR_ANALISE)


In [ ]:
PIPELINE_SOURCE = '#!/usr/bin/env python3\n"""Baixa do zero os ZIPs oficiais de dengue e consolida um CSV ordenado por ano de origem."""\nimport argparse\nimport csv\nimport errno\nimport io\nfrom operator import itemgetter\nfrom concurrent.futures import ThreadPoolExecutor, as_completed\nfrom html.parser import HTMLParser\nfrom http.client import IncompleteRead\nimport json\nfrom pathlib import Path, PurePosixPath\nimport re\nimport shutil\nimport sys\nimport time\nfrom urllib.error import HTTPError\nfrom urllib.parse import unquote, urlparse\nfrom urllib.request import Request, urlopen\nimport zipfile\nfrom contextlib import contextmanager\n\nCATALOG = \'https://dadosabertos.saude.gov.br/dataset/arboviroses-dengue\'\nDEFAULT_OUTPUT = Path(__file__).resolve().parents[1] / \'data/sinan-arquivos\'\nBLOCK = 1024 * 1024\n\n\ndef request(url, method=\'GET\', headers=None):\n    return urlopen(Request(url, method=method, headers={\'User-Agent\': \'Denguinhos-dataset-downloader/1.0\', **(headers or {})}), timeout=120)\n\n\ndef retry(operation):\n    for attempt in range(6):\n        try:\n            return operation()\n        except (OSError, EOFError, IncompleteRead) as error:\n            if isinstance(error, OSError) and error.errno in (errno.ENOSPC, errno.EDQUOT, errno.EACCES):\n                raise\n            if isinstance(error, HTTPError) and error.code not in (408, 429, 500, 502, 503, 504):\n                raise\n            if attempt == 5:\n                raise\n            delay = min(30, 2 ** attempt)\n            print(f\'Falha temporária: {error}; nova tentativa em {delay}s\', flush=True)\n            time.sleep(delay)\n\n\nclass CatalogParser(HTMLParser):\n    def __init__(self):\n        super().__init__()\n        self.capturing = False\n        self.parts = []\n\n    def handle_starttag(self, tag, attrs):\n        if tag == \'script\' and dict(attrs).get(\'id\') == \'__NEXT_DATA__\':\n            self.capturing = True\n\n    def handle_data(self, data):\n        if self.capturing:\n            self.parts.append(data)\n\n    def handle_endtag(self, tag):\n        if tag == \'script\':\n            self.capturing = False\n\n\ndef resources_from_html(html):\n    parser = CatalogParser()\n    parser.feed(html)\n    payload = json.loads(\'\'.join(parser.parts))\n    resources = payload[\'props\'][\'pageProps\'][\'resources\']\n    selected = {}\n    for resource in resources:\n        url = resource.get(\'url\', \'\')\n        path = urlparse(url).path.lower()\n        if resource.get(\'state\', \'active\') != \'active\':\n            continue\n        if resource.get(\'format\', \'\').upper() != \'CSV\' and not path.endswith((\'.csv\', \'.csv.zip\')):\n            continue\n        if urlparse(url).scheme != \'https\':\n            raise ValueError(f\'URL de download inesperada: {url}\')\n        if not path.endswith((\'.zip\', \'.csv\')):\n            raise ValueError(f\'Recurso CSV sem arquivo reconhecido: {url}\')\n        selected[url] = resource\n    if not selected:\n        raise ValueError(\'Nenhum CSV encontrado no catálogo. A estrutura do portal pode ter mudado.\')\n    return list(selected.values())\n\n\ndef load_catalog():\n    def load():\n        with request(CATALOG) as response:\n            return resources_from_html(response.read().decode(\'utf-8\'))\n    return retry(load)\n\n\ndef write_json(path, payload):\n    temporary = path.with_suffix(path.suffix + \'.tmp\')\n    temporary.write_text(json.dumps(payload, ensure_ascii=False, indent=2) + \'\\n\')\n    temporary.replace(path)\n\n\ndef download(resource, folder):\n    """Cada execução e cada tentativa baixa o arquivo inteiro, sem consultar cache."""\n    url = resource[\'url\']\n    name = Path(unquote(urlparse(url).path)).name\n    if name in (\'\', \'.\', \'..\'):\n        raise ValueError(\'Nome de arquivo inválido\')\n    target = folder / name\n    partial = target.with_suffix(target.suffix + \'.part\')\n\n    def transfer():\n        with request(url) as response:\n            if response.status != 200:\n                raise ValueError(f\'Resposta HTTP inesperada: {response.status}\')\n            size_header = response.headers.get(\'Content-Length\')\n            expected = int(size_header) if size_header is not None else None\n            with partial.open(\'wb\') as stream:\n                shutil.copyfileobj(response, stream, BLOCK)\n            size = partial.stat().st_size\n            if expected is not None and size != expected:\n                raise OSError(f\'Download incompleto: {name}\')\n            return {\'url\': url, \'etag\': response.headers.get(\'ETag\'), \'size\': size,\n                    \'modified\': response.headers.get(\'Last-Modified\')}\n    print(f\'Baixando do zero: {name}\', flush=True)\n    identity = retry(transfer)\n    partial.replace(target)\n    return target, identity\n\n\ndef extract(archive, output, identity):\n    destination = output / archive.name.removesuffix(\'.zip\')\n    destination.mkdir(parents=True, exist_ok=True)\n    marker = destination / \'extracao.json\'\n    results = []\n    if archive.suffix.lower() == \'.zip\':\n        with zipfile.ZipFile(archive) as zipped:\n            members = [member for member in zipped.infolist() if not member.is_dir() and member.filename.lower().endswith(\'.csv\')]\n            if not members:\n                raise ValueError(f\'ZIP não contém CSV: {archive.name}\')\n            for member in members:\n                parts = PurePosixPath(member.filename)\n                if parts.is_absolute() or \'..\' in parts.parts or \'\\\\\' in member.filename or \':\' in member.filename:\n                    raise ValueError(f\'Caminho inválido no ZIP: {member.filename}\')\n                target = destination / Path(*parts.parts)\n                target.parent.mkdir(parents=True, exist_ok=True)\n                temporary = target.with_suffix(target.suffix + \'.part\')\n                temporary.unlink(missing_ok=True)\n                if shutil.disk_usage(destination).free < member.file_size:\n                    raise OSError(errno.ENOSPC, f\'Espaço insuficiente para extrair {member.filename} ({member.file_size / BLOCK:.1f} MiB)\')\n                # A leitura até EOF verifica o CRC do ZIP, sem carregar o CSV em memória.\n                try:\n                    with zipped.open(member) as source, temporary.open(\'wb\') as stream:\n                        shutil.copyfileobj(source, stream, BLOCK)\n                except BaseException:\n                    temporary.unlink(missing_ok=True)\n                    raise\n                temporary.replace(target)\n                results.append({\'arquivo\': member.filename, \'bytes\': member.file_size})\n    else:\n        target = destination / archive.name\n        temporary = target.with_suffix(target.suffix + \'.part\')\n        shutil.copyfile(archive, temporary)\n        temporary.replace(target)\n        results.append({\'arquivo\': archive.name, \'bytes\': target.stat().st_size})\n    write_json(marker, {\'origem\': identity, \'csvs\': results})\n    print(f\'Descompactado: {archive.name} ({len(results)} CSV)\', flush=True)\n    return results\n\n\ndef process(resource, output, extract_files=False):\n    archive, identity = download(resource, output / \'zips\')\n    if not extract_files:\n        return {\'recurso\': resource[\'name\'], \'url\': resource[\'url\'], \'arquivo\': str(archive), \'origem\': identity, \'concluido\': True}\n    csvs = extract(archive, output / \'csv\', identity)\n    return {\'recurso\': resource[\'name\'], \'url\': resource[\'url\'], \'arquivo\': str(archive), \'origem\': identity, \'csvs\': csvs, \'concluido\': True}\n\n\ndef resource_year(resource):\n    match = re.search(r\'\\b(20\\d{2}|19\\d{2})\\b\', resource[\'recurso\'])\n    if not match:\n        raise ValueError(f\'Ano não identificado no recurso: {resource["recurso"]}\')\n    return int(match[1])\n\n\n@contextmanager\ndef csv_source(spec, encoding):\n    archive = Path(spec[\'arquivo\'])\n    if spec[\'membro\'] is None:\n        with archive.open(\'r\', encoding=encoding, newline=\'\') as stream:\n            yield stream\n    else:\n        with zipfile.ZipFile(archive) as zipped, zipped.open(spec[\'membro\']) as raw:\n            with io.TextIOWrapper(raw, encoding=encoding, newline=\'\') as stream:\n                yield stream\n\n\ndef inspect_sources(results, encoding):\n    specs, columns = [], [\'ANO_ARQUIVO\']\n    for result in sorted(results, key=lambda item: (resource_year(item), item[\'url\'])):\n        archive = Path(result[\'arquivo\'])\n        if archive.suffix.lower() == \'.zip\':\n            with zipfile.ZipFile(archive) as zipped:\n                members = sorted((i.filename, i.file_size) for i in zipped.infolist() if not i.is_dir() and i.filename.lower().endswith(\'.csv\'))\n        else:\n            members = [(None, archive.stat().st_size)]\n        if not members:\n            raise ValueError(f\'Arquivo sem CSV: {archive}\')\n        for member, size in members:\n            spec = {\'arquivo\': str(archive), \'membro\': member, \'bytes\': size, \'ano\': resource_year(result), \'origem\': result[\'origem\']}\n            with csv_source(spec, encoding) as stream:\n                first_line = stream.readline()\n                try:\n                    delimiter = csv.Sniffer().sniff(first_line, delimiters=\',;\\t\').delimiter\n                except csv.Error:\n                    if any(character in first_line for character in \',;\\t\'):\n                        raise\n                    delimiter = \',\'  # CSV com uma única coluna\n                header = [name.strip().upper() for name in next(csv.reader([first_line], delimiter=delimiter))]\n            if not header or any(not name for name in header) or len(set(header)) != len(header) or \'ANO_ARQUIVO\' in header:\n                raise ValueError(f\'Cabeçalho inválido ou repetido: {archive} / {member}\')\n            spec.update(colunas=header, delimitador=delimiter)\n            specs.append(spec)\n            columns.extend(name for name in header if name not in columns)\n    return specs, columns\n\n\ndef consolidate(results, output, encoding=\'utf-8-sig\'):\n    """Reconstrói integralmente o CSV em fluxo a partir dos downloads desta execução."""\n    output.parent.mkdir(parents=True, exist_ok=True)\n    specs, columns = inspect_sources(results, encoding)\n    manifest_path = output.with_suffix(\'.manifest.json\')\n    temporary = output.with_suffix(output.suffix + \'.part\')\n    state = {\'registros\': 0, \'por_ano_arquivo\': {}}\n    # Limite inferior: união de colunas pode produzir um resultado ainda maior.\n    minimum = sum(spec[\'bytes\'] for spec in specs)\n    if shutil.disk_usage(output.parent).free < minimum:\n        raise OSError(errno.ENOSPC, f\'Consolidação precisa de pelo menos {minimum / 1024**3:.2f} GiB livres; a união de colunas pode exigir mais. Use --output em um disco com espaço.\')\n    with temporary.open(\'w\', encoding=\'utf-8\', newline=\'\', buffering=BLOCK) as stream:\n        writer = csv.writer(stream, lineterminator=\'\\n\')\n        writer.writerow(columns)\n        for index in range(len(specs)):\n            spec = specs[index]\n            source_columns = spec[\'colunas\']\n            positions = {name: position for position, name in enumerate(source_columns)}\n            # itemgetter remapeia as colunas em C, sem um dict por registro.\n            reorder = itemgetter(*(positions.get(name, len(source_columns)) for name in columns[1:]))\n            count = 0\n            year = str(spec[\'ano\'])\n            with csv_source(spec, encoding) as source:\n                reader = csv.reader(source, delimiter=spec[\'delimitador\'], strict=True)\n                next(reader)\n                for row in reader:\n                    if len(row) != len(source_columns):\n                        raise ValueError(f\'Quantidade de campos inválida: {spec["arquivo"]}, linha {reader.line_num}\')\n                    row.append(\'\')\n                    values = reorder(row)\n                    if len(columns) == 2:\n                        values = (values,)\n                    writer.writerow((year,) + values)\n                    count += 1\n            stream.flush()\n            state[\'registros\'] += count\n            state[\'por_ano_arquivo\'][year] = state[\'por_ano_arquivo\'].get(year, 0) + count\n            print(f\'Consolidado {year}: {count:,} registros | fontes: {index + 1}/{len(specs)} | total: {state["registros"]:,}\', flush=True)\n    temporary.replace(output)\n    manifest = {**state, \'bytes\': output.stat().st_size, \'colunas\': columns, \'encoding\': encoding, \'fontes\': specs,\n                \'ordenacao\': \'ANO_ARQUIVO crescente; ordem original dentro de cada arquivo\', \'concluido\': True}\n    write_json(manifest_path, manifest)\n    return manifest\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--output-dir\', type=Path, default=DEFAULT_OUTPUT)\n    parser.add_argument(\'--threads\', type=int, default=4, help=\'Downloads de arquivos simultâneos (padrão: 4).\')\n    parser.add_argument(\'--output\', type=Path, help=\'CSV consolidado (padrão: OUTPUT_DIR/dengue-completo.csv).\')\n    parser.add_argument(\'--encoding\', default=\'utf-8-sig\', help=\'Codificação dos CSVs de origem.\')\n    parser.add_argument(\'--extract-files\', action=\'store_true\', help=\'Também grava cópias individuais descompactadas (usa mais disco).\')\n    parser.add_argument(\'--list-only\', action=\'store_true\', help=\'Lista os recursos sem baixar.\')\n    args = parser.parse_args()\n    if args.threads < 1:\n        parser.error(\'--threads deve ser positivo\')\n    resources = load_catalog()\n    print(f\'Catálogo: {len(resources)} arquivos CSV disponíveis.\', flush=True)\n    if args.list_only:\n        for resource in resources:\n            print(resource[\'name\'], resource[\'url\'])\n        return\n    (args.output_dir / \'zips\').mkdir(parents=True, exist_ok=True)\n    write_json(args.output_dir / \'catalogo.json\', {\'fonte\': CATALOG, \'resources\': resources})\n    print(\'Etapa 1/2: baixando todos os arquivos do catálogo.\', flush=True)\n    results = []\n    with ThreadPoolExecutor(max_workers=args.threads) as pool:\n        futures = {pool.submit(process, resource, args.output_dir, args.extract_files): resource for resource in resources}\n        for future in as_completed(futures):\n            resource = futures[future]\n            try:\n                results.append(future.result())\n            except Exception as error:\n                print(f\'Falha em {resource["name"]}: {error}\', file=sys.stderr, flush=True)\n                results.append({\'recurso\': resource[\'name\'], \'url\': resource[\'url\'], \'concluido\': False, \'erro\': str(error)})\n            write_json(args.output_dir / \'manifesto.json\', {\'fonte\': CATALOG, \'total\': len(resources), \'resultados\': results})\n            print(f\'Progresso: {len(results)}/{len(resources)} arquivos processados\', flush=True)\n    if any(not result[\'concluido\'] for result in results):\n        raise ValueError(\'Alguns arquivos falharam. Execute novamente para baixar tudo do zero.\')\n    output = args.output or args.output_dir / \'dengue-completo.csv\'\n    print(f\'Etapa 2/2: descompactando os ZIPs e reunindo todos os registros em {output}, por ano crescente.\', flush=True)\n    manifest = consolidate(results, output, args.encoding)\n    print(f\'Pipeline concluído: {output} | {manifest["registros"]:,} registros\', flush=True)\n    return output\n\n\nif __name__ == \'__main__\':\n    try:\n        main()\n    except KeyboardInterrupt:\n        print(\'Interrompido. A próxima execução baixará tudo do zero.\', file=sys.stderr)\n        sys.exit(130)\n    except Exception as error:\n        print(f\'Erro: {error}\', file=sys.stderr)\n        sys.exit(1)\n'
SCRIPT.write_text(PIPELINE_SOURCE, encoding="utf-8")


In [ ]:
import importlib.util
import subprocess
import sys

for pacote in ("duckdb", "pandas"):
    if importlib.util.find_spec(pacote) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pacote])

ANALISE_SOURCE = '"""Consultas exatas do SINAN em Parquet, sem carregar o dataset inteiro em pandas."""\nfrom itertools import combinations_with_replacement\nfrom pathlib import Path\nimport os\nimport re\nimport time\n\nAUSENCIAS = [\'\', \'nan\', \'NaN\', \'NAN\', \'null\', \'NULL\', \'None\', \'none\', \'NaT\', \'<NA>\']\nUFS = [\n    (\'11\',\'RO\',\'Norte\'), (\'12\',\'AC\',\'Norte\'), (\'13\',\'AM\',\'Norte\'),\n    (\'14\',\'RR\',\'Norte\'), (\'15\',\'PA\',\'Norte\'), (\'16\',\'AP\',\'Norte\'), (\'17\',\'TO\',\'Norte\'),\n    (\'21\',\'MA\',\'Nordeste\'), (\'22\',\'PI\',\'Nordeste\'), (\'23\',\'CE\',\'Nordeste\'),\n    (\'24\',\'RN\',\'Nordeste\'), (\'25\',\'PB\',\'Nordeste\'), (\'26\',\'PE\',\'Nordeste\'),\n    (\'27\',\'AL\',\'Nordeste\'), (\'28\',\'SE\',\'Nordeste\'), (\'29\',\'BA\',\'Nordeste\'),\n    (\'31\',\'MG\',\'Sudeste\'), (\'32\',\'ES\',\'Sudeste\'), (\'33\',\'RJ\',\'Sudeste\'), (\'35\',\'SP\',\'Sudeste\'),\n    (\'41\',\'PR\',\'Sul\'), (\'42\',\'SC\',\'Sul\'), (\'43\',\'RS\',\'Sul\'),\n    (\'50\',\'MS\',\'Centro-Oeste\'), (\'51\',\'MT\',\'Centro-Oeste\'), (\'52\',\'GO\',\'Centro-Oeste\'), (\'53\',\'DF\',\'Centro-Oeste\'),\n]\n\n\ndef qi(value):\n    return \'"\' + value.replace(\'"\', \'""\') + \'"\'\n\n\ndef qs(value):\n    return "\'" + str(value).replace("\'", "\'\'") + "\'"\n\n\nclass AnaliseSinan:\n    def __init__(self, pasta, memoria=\'3GB\', threads=None):\n        import duckdb\n        self.pasta = Path(pasta)\n        self.pasta.mkdir(parents=True, exist_ok=True)\n        self.parquet = self.pasta / \'dengue-analitico.parquet\'\n        self.con = duckdb.connect()\n        self.con.execute(f\'SET threads = {int(threads or min(8, os.cpu_count() or 2))}\')\n        self.con.execute(f\'SET memory_limit = {qs(memoria)}\')\n        self.con.execute(\'SET preserve_insertion_order = false\')\n        self.con.execute(f"SET temp_directory = {qs(self.pasta / \'temporarios\')}")\n        self.pronta = False\n\n    def fechar(self):\n        self.con.close()\n        self.pronta = False\n\n    def _verificar(self):\n        if not self.pronta:\n            raise RuntimeError(\'Execute preparar_csv() ou abrir_parquet() antes das consultas.\')\n\n    def preparar_csv(self, arquivo):\n        """Sempre reconverte o CSV escolhido; não verifica cache nem pula downloads."""\n        inicio = time.perf_counter()\n        arquivo = Path(arquivo)\n        nulls = \'[\' + \',\'.join(qs(value) for value in AUSENCIAS) + \']\'\n        self.pronta = False\n        self.con.execute(f"""CREATE OR REPLACE TEMP VIEW origem AS\n            SELECT * FROM read_csv({qs(arquivo)}, header=true, delim=\',\',\n                all_varchar=true, sample_size=2048, parallel=true,\n                nullstr={nulls}, strict_mode=true, ignore_errors=false)""")\n        columns = [row[0] for row in self.con.execute(\'DESCRIBE origem\').fetchall()]\n        if any(name.startswith(\'_SINAN_\') for name in columns):\n            raise ValueError(\'A origem contém nomes reservados _SINAN_.\')\n        if \'SG_UF\' not in columns and \'ID_MN_RESI\' not in columns:\n            raise ValueError(\'A origem precisa de SG_UF ou ID_MN_RESI (residência).\')\n        if \'ANO_ARQUIVO\' not in columns:\n            raise ValueError(\'Use o CSV consolidado com ANO_ARQUIVO.\')\n        temporary = self.parquet.with_suffix(\'.parquet.part\')\n        temporary.unlink(missing_ok=True)\n        print(\'Preparando a base inteira: CSV → Parquet. Esta etapa lê o CSV uma vez.\', flush=True)\n        try:\n            self.con.execute(f"""COPY (SELECT * FROM origem) TO {qs(temporary)}\n                (FORMAT PARQUET, COMPRESSION ZSTD, COMPRESSION_LEVEL 1, ROW_GROUP_SIZE 122880)""")\n            temporary.replace(self.parquet)\n        except BaseException:\n            temporary.unlink(missing_ok=True)\n            raise\n        self.abrir_parquet(self.parquet)\n        print(f\'Preparação concluída: {self.total:,} registros em {time.perf_counter()-inicio:.1f}s.\', flush=True)\n        return self\n\n    def abrir_parquet(self, arquivo):\n        """Abre explicitamente um artefato já preparado; não reconverte CSV."""\n        self.pronta = False\n        self.parquet = Path(arquivo)\n        self.con.execute(f\'CREATE OR REPLACE TEMP VIEW origem_parquet AS SELECT * FROM read_parquet({qs(self.parquet)})\')\n        columns = [row[0] for row in self.con.execute(\'DESCRIBE origem_parquet\').fetchall()]\n        if \'ANO_ARQUIVO\' not in columns or not {\'SG_UF\',\'ID_MN_RESI\'}.intersection(columns):\n            raise ValueError(\'O Parquet precisa de ANO_ARQUIVO e localização de residência.\')\n        if any(c.startswith(\'_SINAN_\') for c in columns):\n            raise ValueError(\'O Parquet contém nomes de colunas reservados.\')\n        self.colunas = columns\n        geo = self._geografia_sql(columns)\n        dates = []\n        for raw, derived in [(\'DT_NOTIFIC\',\'_SINAN_DT_NOTIFIC\'), (\'DT_SIN_PRI\',\'_SINAN_DT_SIN_PRI\')]:\n            expression = f"try_strptime(trim({qi(raw)}), \'%Y-%m-%d\')::DATE" if raw in columns else \'NULL::DATE\'\n            dates.append(f\'{expression} AS {qi(derived)}\')\n        # Colunas internas são expressões virtuais: só são calculadas quando\n        # a consulta as solicita. Correlações não normalizam geografia/datas.\n        self.con.execute(\'CREATE OR REPLACE TEMP VIEW base AS SELECT *, \' + \',\'.join(geo+dates) + \' FROM origem_parquet\')\n        self.total = self.con.execute(\'SELECT count(*) FROM origem_parquet\').fetchone()[0]\n        print(\'Criando resumo geográfico exato, lendo apenas localização e ano...\', flush=True)\n        keys = [\'ANO_ARQUIVO\'] + [c for c in (\'SG_UF\',\'ID_MN_RESI\') if c in columns]\n        self.con.execute(\'CREATE OR REPLACE TEMP TABLE geo_agregada AS SELECT \' + \',\'.join(qi(c) for c in keys)\n            + \',count(*) AS quantidade FROM origem_parquet GROUP BY ALL\')\n        # Agrupa primeiro e só depois normaliza os códigos distintos.\n        # Evita regex/mapear UF 32 milhões de vezes durante a preparação.\n        self.con.execute(\'CREATE OR REPLACE TEMP TABLE resumo_geo AS SELECT ANO_ARQUIVO AS ano, \'\n            \'_SINAN_MUN_RES AS municipio,_SINAN_UF_RES AS uf,_SINAN_REGIAO AS regiao,\'\n            \'sum(quantidade)::BIGINT AS notificacoes FROM (SELECT *, \' + \',\'.join(geo)\n            + \' FROM geo_agregada) GROUP BY ALL\')\n        self.con.execute(\'DROP TABLE geo_agregada\')\n        assert self.con.execute(\'SELECT coalesce(sum(notificacoes),0) FROM resumo_geo\').fetchone()[0] == self.total\n        self.pronta = True\n        return self\n\n    def _geografia_sql(self, columns):\n        uf = "regexp_replace(upper(trim(\\"SG_UF\\")), \'\\\\.0+$\', \'\')" if \'SG_UF\' in columns else \'NULL::VARCHAR\'\n        mun = "regexp_replace(trim(\\"ID_MN_RESI\\"), \'\\\\.0+$\', \'\')" if \'ID_MN_RESI\' in columns else \'NULL::VARCHAR\'\n        codes = \',\'.join(qs(row[0]) for row in UFS)\n        municipality = f"CASE WHEN regexp_full_match({mun}, \'[0-9]{{6,7}}\') AND left({mun},2) IN ({codes}) THEN left({mun},6) END"\n        uf_cases = \' \'.join(f\'WHEN {qs(alias)} THEN {qs(code)}\' for code,sigla,_ in UFS for alias in (code,sigla))\n        uf_code = f\'coalesce(CASE {uf} {uf_cases} END, left({municipality},2))\'\n        region_cases = \' \'.join(f\'WHEN {qs(code)} THEN {qs(region)}\' for code,_,region in UFS)\n        region = f"CASE {uf_code} {region_cases} ELSE \'Não identificada\' END"\n        return [f\'{municipality} AS _SINAN_MUN_RES\', f\'{uf_code} AS _SINAN_UF_RES\', f\'{region} AS _SINAN_REGIAO\']\n\n    def por_ano(self):\n        self._verificar()\n        return self.con.execute(\'SELECT ano,sum(notificacoes)::BIGINT AS notificacoes FROM resumo_geo GROUP BY ano ORDER BY ano\').df()\n\n    def municipio(self, codigo):\n        self._verificar()\n        codigo = str(codigo).strip()\n        if not re.fullmatch(r\'[0-9]{6,7}\', codigo):\n            raise ValueError(\'Informe um código municipal de seis ou sete dígitos.\')\n        # Retorna também os anos com zero notificações desse município na base.\n        return self.con.execute(\'\'\'SELECT ano,\n            sum(CASE WHEN municipio=? THEN notificacoes ELSE 0 END)::BIGINT AS notificacoes\n            FROM resumo_geo GROUP BY ano ORDER BY ano\'\'\', [codigo[:6]]).df()\n\n    def regioes(self):\n        self._verificar()\n        return self.con.execute(\'\'\'SELECT regiao, sum(notificacoes)::BIGINT AS notificacoes,\n            100.0*sum(notificacoes)/nullif(?,0) AS percentual_do_total\n            FROM resumo_geo GROUP BY regiao ORDER BY notificacoes DESC,regiao\'\'\', [self.total]).df()\n\n    def ausencias(self):\n        self._verificar()\n        # As estatísticas do Parquet dão ausências exatas sem varrer 6,4 bilhões de células.\n        rows = self.con.execute(\'\'\'SELECT path_in_schema, sum(stats_null_count)::BIGINT,\n            bool_and(stats_null_count IS NOT NULL)\n            FROM parquet_metadata(?) GROUP BY column_id,path_in_schema ORDER BY column_id\'\'\', [str(self.parquet)]).fetchall()\n        result = []\n        for name, missing, complete in rows:\n            if name not in self.colunas:\n                continue\n            if not complete:\n                missing = self.con.execute(f\'SELECT count(*)-count({qi(name)}) FROM base\').fetchone()[0]\n            result.append((name, missing, 100*missing/self.total if self.total else 0))\n        import pandas as pd\n        return pd.DataFrame(result, columns=[\'coluna\',\'ausentes\',\'percentual_ausente\']).sort_values(\'percentual_ausente\', ascending=False)\n\n    def frequencias(self, campo):\n        self._verificar()\n        if campo not in self.colunas:\n            raise ValueError(f\'Coluna não encontrada: {campo}\')\n        return self.con.execute(f\'SELECT {qi(campo)} AS valor,count(*) AS notificacoes FROM base GROUP BY 1 ORDER BY 2 DESC\').df()\n\n    def tabela_associacao(self, campo_a, campo_b):\n        self._verificar()\n        if campo_a not in self.colunas or campo_b not in self.colunas:\n            raise ValueError(\'Selecione colunas existentes.\')\n        result = self.con.execute(f\'\'\'SELECT {qi(campo_a)} AS categoria_a,\n            {qi(campo_b)} AS categoria_b,count(*) AS quantidade FROM base\n            WHERE {qi(campo_a)} IS NOT NULL AND {qi(campo_b)} IS NOT NULL\n            GROUP BY 1,2\'\'\').df()\n        return result.pivot_table(index=\'categoria_a\', columns=\'categoria_b\',\n            values=\'quantidade\', aggfunc=\'sum\', fill_value=0).astype(\'int64\')\n\n    def serie_temporal(self, municipio=None, data=\'DT_NOTIFIC\', periodo=\'month\'):\n        self._verificar()\n        if data not in (\'DT_NOTIFIC\',\'DT_SIN_PRI\') or periodo not in (\'day\',\'week\',\'month\',\'year\'):\n            raise ValueError(\'Escolha DT_NOTIFIC/DT_SIN_PRI e day/week/month/year.\')\n        column = qi(\'_SINAN_\' + data)\n        filter_sql = \'\'\n        params = []\n        if municipio is not None:\n            codigo = str(municipio).strip()\n            if not re.fullmatch(r\'[0-9]{6,7}\', codigo):\n                raise ValueError(\'Código municipal inválido.\')\n            filter_sql = \' AND _SINAN_MUN_RES=?\'\n            params = [codigo[:6]]\n        return self.con.execute(f\'\'\'SELECT date_trunc(\'{periodo}\',{column})::DATE AS periodo,\n            count(*) AS notificacoes FROM base WHERE {column} IS NOT NULL {filter_sql}\n            GROUP BY 1 ORDER BY 1\'\'\', params).df()\n\n    def qualidade_datas(self):\n        self._verificar()\n        expressions = ["count(*) FILTER (WHERE _SINAN_DT_NOTIFIC < _SINAN_DT_SIN_PRI) AS notificacao_antes_dos_sintomas"]\n        for field in (\'DT_NOTIFIC\',\'DT_SIN_PRI\'):\n            if field in self.colunas:\n                expressions.append(f\'count(*) FILTER (WHERE {qi(field)} IS NOT NULL AND {qi("_SINAN_"+field)} IS NULL) AS {qi(field+"_INVALIDA")}\')\n        return self.con.execute(\'SELECT \' + \',\'.join(expressions) + \' FROM base\').df()\n\n    def correlacoes(self, campos, minimo_validos=500):\n        self._verificar()\n        campos = list(dict.fromkeys(campos))\n        if not campos or any(c not in self.colunas for c in campos):\n            raise ValueError(\'Selecione apenas colunas existentes.\')\n        if minimo_validos < 2:\n            raise ValueError(\'minimo_validos deve ser pelo menos 2.\')\n        expressions = [f"CASE WHEN trim({qi(c)}) IN (\'1\',\'1.0\') THEN 1.0 WHEN trim({qi(c)}) IN (\'2\',\'2.0\') THEN 0.0 END AS x_{i}" for i,c in enumerate(campos)]\n        pairs = list(combinations_with_replacement(range(len(campos)), 2))\n        aggregates = []\n        for k,(i,j) in enumerate(pairs):\n            aggregates += [f\'corr(x_{i},x_{j}) AS r_{k}\', f\'count(x_{i}+x_{j}) AS n_{k}\']\n        values = self.con.execute(\'WITH binarios AS (SELECT \' + \',\'.join(expressions) + \' FROM base) SELECT \' + \',\'.join(aggregates) + \' FROM binarios\').fetchone()\n        import pandas as pd\n        corr = pd.DataFrame(float(\'nan\'), index=campos, columns=campos)\n        valid = pd.DataFrame(0, index=campos, columns=campos)\n        for k,(i,j) in enumerate(pairs):\n            coefficient, count = values[2*k:2*k+2]\n            corr.iloc[i,j] = corr.iloc[j,i] = coefficient if count >= minimo_validos and coefficient is not None else float(\'nan\')\n            valid.iloc[i,j] = valid.iloc[j,i] = count\n        return corr, valid\n\n    def duplicatas_exatas(self):\n        self._verificar()\n        # Comparação exata dos campos publicados; sem hash probabilístico.\n        cols = \',\'.join(qi(c) for c in self.colunas if c != \'ANO_ARQUIVO\')\n        return self.con.execute(f\'\'\'SELECT coalesce(sum(n-1),0)::BIGINT AS copias_excedentes,\n            coalesce(sum(n),0)::BIGINT AS linhas_envolvidas, count(*) AS grupos_duplicados\n            FROM (SELECT {cols},count(*) AS n FROM base GROUP BY ALL HAVING count(*)>1)\'\'\').df()\n'
namespace_analise = {"__name__": "sinan_analise_completa"}
exec(compile(ANALISE_SOURCE, "sinan_analise_completa.py", "exec"), namespace_analise)
AnaliseSinan = namespace_analise["AnaliseSinan"]


## Drive opcional

Ative a opção na configuração para montar o Drive e salvar a saída ao final. Para um CSV já salvo no Drive, você também pode montá-lo manualmente e ajustar `CSV_FINAL`. Prefira o Parquet local para as consultas: ler pelo Drive pode acrescentar latência.

Os arquivos de `/content` são temporários. A cópia para o Drive requer quota disponível. [Documentação oficial do Colab](https://colab.research.google.com/notebooks/io.ipynb).


In [ ]:
if SALVAR_NO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    relativo = Path(PASTA_NO_DRIVE)
    if relativo.is_absolute() or ".." in relativo.parts:
        raise ValueError("PASTA_NO_DRIVE deve ser relativa a Meu Drive")
    DESTINO_DRIVE = Path("/content/drive/MyDrive") / relativo
    DESTINO_DRIVE.mkdir(parents=True, exist_ok=True)


## 1. Download e CSV único — somente quando solicitado

Com `EXECUTAR_DOWNLOAD = True`, baixa todos os ZIPs novamente e gera o CSV por ano de origem crescente. Não consulta cache nem retoma downloads. Com a opção desativada, esta etapa não abre nem regrava os dados existentes.


In [ ]:
import subprocess
import sys
import time

PIPELINE_OK = False
if EXECUTAR_DOWNLOAD:
    inicio = time.monotonic()
    comando = [sys.executable, "-u", str(SCRIPT), "--output-dir", str(DADOS),
               "--output", str(CSV_FINAL), "--threads", str(THREADS_DOWNLOAD)]
    with subprocess.Popen(comando, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, encoding="utf-8", errors="replace", bufsize=1) as processo:
        try:
            for linha in processo.stdout:
                print(linha, end="", flush=True)
            codigo = processo.wait()
        except BaseException:
            processo.terminate()
            processo.wait()
            raise
    if codigo:
        raise RuntimeError(f"Download/consolidação falhou: código {codigo}")
    PIPELINE_OK = True
    print(f"Tempo de download e consolidação: {(time.monotonic()-inicio)/60:.1f} min")
else:
    print("Download desativado. Será usado o arquivo indicado em CSV_FINAL.")


In [ ]:
import json

MANIFESTO = CSV_FINAL.with_suffix(".manifest.json")
if EXECUTAR_DOWNLOAD:
    if not PIPELINE_OK:
        raise RuntimeError("O download não terminou com sucesso")
    resumo = json.loads(MANIFESTO.read_text())
    print(f"CSV: {CSV_FINAL} | registros: {resumo['registros']:,}")
else:
    print("Fonte selecionada para análise:", CSV_FINAL)


## 2. Preparação única da base completa

Converte o CSV para Parquet comprimido, com memória limitada e processamento nativo. Preserva todas as linhas e as colunas originais. Cinco colunas internas são expressões virtuais, calculadas somente quando uma consulta precisa delas. O resumo geográfico agrupa antes de normalizar os códigos, evitando transformar milhões de valores repetidos. Strings técnicas de ausência (`nan`, vazios etc.) tornam-se nulos na cópia analítica; o CSV original não é alterado.

Erros de parsing interrompem a preparação: nenhuma linha é ignorada silenciosamente. Um resumo por ano, município e região é calculado uma vez sobre **todos os registros**. Depois disso, reexecute as células de consulta sem rodar novamente esta preparação.

Para abrir um Parquet preparado em outra sessão, selecione `PREPARAR_ANALISE = False` e configure `PARQUET_EXISTENTE`. A primeira conversão requer espaço adicional para o Parquet e temporários. A ordem das linhas do Parquet é livre; o CSV continua em ordem anual e as séries são ordenadas pela data escolhida.


In [ ]:
if "analise" in globals():
    analise.fechar()
analise = AnaliseSinan(PASTA_ANALISE, memoria=MEMORIA_ANALISE, threads=THREADS_ANALISE)
if PREPARAR_ANALISE:
    analise.preparar_csv(CSV_FINAL)
else:
    analise.abrir_parquet(PARQUET_EXISTENTE)
print(f"BASE COMPLETA: {analise.total:,} registros | {len(analise.colunas)} colunas de origem")
print(f"Parquet: {analise.parquet} | {analise.parquet.stat().st_size / 1024**3:.2f} GiB")
print("Não foi criado um DataFrame com a base inteira.")


## 3. Totais e ausências exatas

Os totais por ano usam o resumo calculado na preparação. As ausências são obtidas das estatísticas do Parquet, evitando ler bilhões de valores novamente. Cardinalidade e contagem de valores distintos de todas as colunas não são calculadas automaticamente.


In [ ]:
from IPython.display import display
import matplotlib.pyplot as plt

por_ano = analise.por_ano()
perfil_completo = analise.ausencias()
display(por_ano)
display(perfil_completo.head(30).round(2))
perfil_completo.head(20).sort_values("percentual_ausente").plot.barh(
    x="coluna", y="percentual_ausente", figsize=(10,6), legend=False,
    title="Ausências — base completa")
plt.xlabel("% dos registros")
plt.tight_layout(); plt.show()


## 4. Município e regiões — base inteira

Usa município de residência (`ID_MN_RESI`), normalizado para seis dígitos. A região vem de `SG_UF` de residência; se inválida, tenta a UF do município. Localização não identificada permanece no denominador. São **notificações**, sem filtro de confirmação e sem remover duplicatas: os percentuais não são incidência populacional.


In [ ]:
por_municipio = analise.municipio(CODIGO_MUNICIPIO)
print(f"Município {CODIGO_MUNICIPIO}: {int(por_municipio['notificacoes'].sum()):,} notificações")
display(por_municipio)


In [ ]:
resultado_regioes = analise.regioes()
display(resultado_regioes.round(2))
resultado_regioes.plot.bar(x="regiao", y="percentual_do_total", legend=False,
    figsize=(10,5), title="Participação nas notificações — base completa")
plt.ylabel("% do total, incluindo localização não identificada")
plt.xticks(rotation=25)
plt.tight_layout(); plt.show()


## 5. Série temporal e qualidade das datas

Agrega por data de notificação ou sintomas, não pelo ano do ZIP. Cada consulta lê somente as colunas necessárias do Parquet. Para série de um município, passe seu código; para o Brasil, use `None`. Datas inválidas são contabilizadas na tabela de qualidade e não entram na série. Períodos ausentes não são preenchidos automaticamente com zero.


In [ ]:
serie = analise.serie_temporal(municipio=CODIGO_MUNICIPIO,
    data=DATA_SERIE, periodo=PERIODO_SERIE)
display(serie.head(12))
if not serie.empty:
    serie.plot(x="periodo", y="notificacoes", figsize=(14,4), legend=False,
        title=f"Notificações por {DATA_SERIE} — município {CODIGO_MUNICIPIO}")
    plt.tight_layout(); plt.show()
display(analise.qualidade_datas())


## 6. Correlações exatas — campos selecionados

Usa todos os registros com dados válidos para cada par. Apenas nos indicadores cujo dicionário confirma `1=sim` e `2=não`, calcula coeficiente phi e número de observações por par. Outros códigos ficam nulos para esse cálculo. A consulta lê somente os indicadores escolhidos; cinco campos têm 10 pares distintos, enquanto 17 têm 136. Ampliar a lista aumenta o trabalho. Correlação não demonstra causalidade e mudanças no formulário entre anos afetam a interpretação.


In [ ]:
if CALCULAR_CORRELACOES:
    campos = [c for c in CAMPOS_CORRELACAO if c in analise.colunas]
    if campos:
        correlacoes, observacoes_validas = analise.correlacoes(campos)
        display(correlacoes.round(3))
        display(observacoes_validas)
    else:
        print("Nenhum dos indicadores escolhidos está disponível.")
else:
    print("Correlações desativadas na configuração.")


## Associação categórica — base completa

A tabela de contingência é agregada no DuckDB lendo apenas duas colunas; somente a tabela pequena chega ao SciPy. O teste inclui as categorias não nulas publicadas, inclusive códigos de ignorado; revise-os conforme o dicionário. Observe o V de Cramér e frequências esperadas, não apenas o p-valor. A análise agregada mistura anos e não ajusta fatores de confusão.


In [ ]:
if {"CS_SEXO", "CLASSI_FIN"}.issubset(analise.colunas):
    from scipy.stats import chi2_contingency
    tabela = analise.tabela_associacao("CS_SEXO", "CLASSI_FIN")
    display(tabela)
    if min(tabela.shape) >= 2:
        chi2, p, _, esperadas = chi2_contingency(tabela, correction=False)
        n = tabela.to_numpy().sum()
        v = (chi2 / (n * (min(tabela.shape)-1))) ** 0.5
        print(f"p={p:.3g}; V de Cramér={v:.3f}; esperadas <5: {(esperadas<5).mean():.1%}")


## 7. Duplicatas exatas — operação pesada, optativa

Compara todos os campos originais, ignorando `ANO_ARQUIVO` e colunas internas. Não usa aproximações por hash. Em 32,5 milhões de registros e 197 colunas, pode exigir bastante tempo e espaço temporário. Fica desativada por padrão. Registros idênticos nas colunas publicadas podem representar notificações diferentes; nenhum registro é removido automaticamente.


In [ ]:
if CALCULAR_DUPLICATAS_EXATAS:
    display(analise.duplicatas_exatas())
else:
    print("Duplicatas exatas desativadas. Ative conscientemente na configuração.")


In [ ]:
if not globals().get("PIPELINE_OK", False) and globals().get("EXECUTAR_DOWNLOAD", True):
    raise RuntimeError("Pipeline não concluído; o resultado não será copiado.")

if SALVAR_NO_DRIVE:
    for origem in (CSV_FINAL, MANIFESTO):
        destino = DESTINO_DRIVE / origem.name
        temporario = destino.with_suffix(destino.suffix + ".part")
        print(f"Copiando {origem.name} para o Drive...", flush=True)
        with origem.open("rb") as entrada, temporario.open("wb") as saida:
            shutil.copyfileobj(entrada, saida, 8 * 1024 * 1024)
        temporario.replace(destino)
else:
    print("Cópia de CSV para o Drive desativada.")


In [ ]:
if SALVAR_NO_DRIVE:
    origem = analise.parquet
    destino = DESTINO_DRIVE / origem.name
    temporario = destino.with_suffix(destino.suffix + ".part")
    print("Copiando Parquet para o Drive...", flush=True)
    with origem.open("rb") as entrada, temporario.open("wb") as saida:
        shutil.copyfileobj(entrada, saida, 8 * 1024 * 1024)
    temporario.replace(destino)
    print("Base analítica persistida:", destino)


## Treinamento e consultas adicionais

A base completa é a view SQL `base` da conexão `analise.con`; ela não é a variável `dados` das células antigas. Somente resultados pequenos são convertidos para pandas. Para frequências de uma coluna, use `analise.frequencias("CLASSI_FIN")`. Para a série nacional, use `analise.serie_temporal(municipio=None, data=DATA_SERIE, periodo=PERIODO_SERIE)`.

O DataFrame `serie` pode ser usado para preparar o treinamento temporal, depois de definir o alvo (notificações, casos prováveis ou confirmados), cobertura e critérios de preenchimento de períodos. Divida treino e teste cronologicamente. Evite `SELECT * FROM base` seguido de `.df()`: isso volta a carregar dezenas de milhões de linhas na RAM.

Os arquivos e DataFrames antigos não são apagados. As antigas células de prévia foram retiradas deste notebook para não confundir resultados de amostra com a base inteira. O notebook original está preservado em `notebooks/referencia/sinan_dengue_colab_antes_refatoracao.ipynb` no projeto.
